#### 감정 분석 자연어 처리
1. data 폴더 안에 ratings_train.txt 파일을 로드
2. 데이터를 상위 500개 데이터만 추출
3. 리뷰 데이터와 감정 데이터로 나눠준다.
4. 리뷰 데이터를 토큰화(komoran함수 이용) -> 벡터화(Word2Vec, 단위 벡터의 평균)
5. Word2Vec 학습
    - window -> 3
    - epochs -> 10
    - min_count -> 5
    - sg -> 1
    - seed -> 42
6. 벡터화(Word2Vec, 단위 벡터의 평균)
7. 분류 모델(SVC, Logistic)
8. train, test을 이용하여 2개의 모델 중 성능이 높은 모델이 무엇인가?
9. 단위 벡터의 평균의 성능과 단위 벡터 + 중요도 평균의 성능의 차이를 확인

In [68]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from konlpy.tag import Komoran
from sklearn.linear_model import LogisticRegression

In [69]:
df = pd.read_csv('../data/ratings_train.txt', sep='\t')
df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [70]:
df = df.head(500)
df

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1
...,...,...,...
495,10049872,그냥 기독교영화네요. 좀 더 깊이있는 내용을 기대했는데.. 실망입니다. 영화도 뭔가...,0
496,9427493,그냥 책으로 읽는게 더 절절하게 다가온다. 영화는 내용을 너무 비약하고 삭제해서 행...,0
497,6629097,너무나 감동적인 영화,1
498,9995177,스킨헤드성님들이 이 영화를 싫어합니다.,0


In [71]:
X = df['document'].values
Y = df['label'].values

In [72]:
import mod

In [73]:
tokenize = mod.bulid_tokenize()

In [74]:
X_tokens = []
for raw in X:
    X_tokens.append(tokenize(raw))
X_tokens

[['더빙', '진짜', '짜증', '나', '목소리'],
 ['포스터', '초딩', '영화', '오버', '연기', '가볍'],
 [],
 ['교도소', '이야기', '솔직히', '재미', '없', '평점', '조정'],
 ['익살', '연기', '돋보이', '영화', '스파이더맨', '늙', '보이', '하', '커스틴 던스트', '너무나'],
 ['막', '걸음마', '떼', '초등학교', '학년', '아깝'],
 ['원작', '긴장감', '제대로', '살리'],
 ['반개',
  '아깝',
  '욕',
  '나오',
  '이응경',
  '길용우',
  '연기',
  '생활',
  '이',
  '정말',
  '발로',
  '납치',
  '감금',
  '반복',
  '반복',
  '이',
  '드라마',
  '가족',
  '없',
  '연기',
  '못하',
  '사람',
  '모이'],
 ['액션', '없', '재미', '있', '안', '영화'],
 ['왜', '평점', '낮', '꽤', '보', '헐리우드', '너무', '길들이'],
 [],
 ['볼',
  '때',
  '눈물',
  '나서',
  '죽',
  '향수',
  '자극',
  '!!',
  '허진호',
  '감성',
  '절제',
  '멜로',
  '달인',
  '이다'],
 ['울', '손들', '횡단보도', '건너', '때', '뛰쳐나오', '이범수', '연기', '드럽'],
 ['좋', '신문', '기사', '로만', '보다', '보', '자꾸', '잊어버리', '사람'],
 ['취향',
  '존중',
  '진짜',
  '극장',
  '보',
  '영화',
  '가장',
  '노',
  '재',
  '노',
  '감동',
  '스토리',
  '어거지',
  '감동',
  '어거지'],
 ['매번', '긴장'],
 ['참',
  '사람',
  '웃기',
  '바스코',
  '이기',
  '락스',
  '코',
  '까',
  '고',
  '바비',
  '이기',
  '아이돌',
  '

In [75]:
w2v = Word2Vec(
    sentences=X_tokens,
    window=3,
    min_count=3,
    sg = 1,
    epochs=5,
    seed=42
)

In [76]:
wv = w2v.wv

In [77]:
def sent_embed_mean(tokens):
    vecs = []
    for word in tokens:
        if word in wv.index_to_key:
            vecs.append(wv[word])
    result =  np.mean(vecs, axis=0) if vecs else np.zeros(wv.vector_size)
    return result

In [78]:
X_embed = [sent_embed_mean(token) for token in X_tokens]

In [79]:
svc = SVC(random_state=42)
logistic = LogisticRegression(random_state=42)

In [83]:
def run_model(X, Y, test_size=0.2, model='svc'):
    # X는 독립변수
    # Y는 종속변수
    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y, test_size=test_size, random_state=42, stratify=Y
    )
    svc = SVC(random_state=42)
    logi = LogisticRegression(random_state=42)
    # 모델에 학습
    if model == 'svc':
        svc.fit(X_train, Y_train)
        # 학습된 모델에 예측 값
        y_pred = svc.predict(X_test)
        print("정확도 :", round(accuracy_score(Y_test, y_pred), 4))
        print("분류 레포트 :", classification_report(Y_test, y_pred))
    elif model == 'logistic':
        logi.fit(X_train, Y_train)
        # 학습된 모델에 예측 값
        y_pred = logi.predict(X_test)
        print("정확도 :", round(accuracy_score(Y_test, y_pred), 4))
        print("분류 레포트 :", classification_report(Y_test, y_pred))

In [84]:
run_model(X_embed, Y)

정확도 : 0.6
분류 레포트 :               precision    recall  f1-score   support

           0       0.58      0.58      0.58        48
           1       0.62      0.62      0.62        52

    accuracy                           0.60       100
   macro avg       0.60      0.60      0.60       100
weighted avg       0.60      0.60      0.60       100



In [85]:
run_model(X_embed, Y, model='logistic')

정확도 : 0.52
분류 레포트 :               precision    recall  f1-score   support

           0       0.00      0.00      0.00        48
           1       0.52      1.00      0.68        52

    accuracy                           0.52       100
   macro avg       0.26      0.50      0.34       100
weighted avg       0.27      0.52      0.36       100



c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

In [ ]:
def predict_sentence_list(sentences, model):
    # sentences : 문장들의 리스트
    # 문장들을 토큰화 -> 임베딩
    X_test = []
    for sent in sentences:
        # token() 함수를 호출하여 토큰화
        tokens = tokenize(sent)
        # 토큰화된 문장을 sent_embed_mean 함수에 입력하여 호출(단위 벡터의 평균)
        vec = sent_embed_mean(tokens)
        X_test.append(vec)
    
    preds = model.predict(X_test)
    result = []
    for sent, pred in zip(sentences, preds):
        label = '긍정' if pred == 1 else '부정'
        result.append([sent, label])
    return result

In [86]:
def sent_embed_tfidf(tokens):
    vecs = []
    weight = []
    for word in tokens:
        # tokens의 각각의 단어가 Word2Vec과 TF-IDF에 존재한다면
        if word in wv.key_to_index and word in idf:
            # vecs -> 단위벡터와 중요도를 곱한 값을 vecs 추가
            vecs.append(wv[word] * idf[word])
            # weight -> 중요도 데이터를 추가
            weight.append(idf[word])
    # vecs에 데이터가 존재하지 않는다면 -> tokens 안에 단어는 존재하지만 Word2Vec이나 TF-IDF에 단어가 존재하지 않을때
    if not vecs:
        # 희소행렬 되돌려준다. 0행렬
        result = np.zeros(wv.vector_size)
    else:
        result = np.sum(vecs, axis=0) / (np.sum(weight) + 1e-9)
        return result

In [87]:
X_embed2 = [sent_embed_tfidf(token) for token in X_tokens]

NameError: name 'idf' is not defined

In [ ]:
8. train, test을 이용하여 2개의 모델 중 성능이 높은 모델이 무엇인가?
9. 단위 벡터의 평균의 성능과 단위 벡터 + 중요도 평균의 성능의 차이를 확인